# **Mini Project : Grace Natalie Catherine**
## **AI/CV Analyzer with Semantic Search**

In [81]:
# import libraries

import pandas as pd
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics.pairwise import cosine_similarity

import warnings
warnings.filterwarnings('ignore')

# **Business Understanding**

## **Business Problem**
Today's hiring process is highly competitive, and employers need efficient ways to screen and evaluate candidates. Tons of CVs are submitted for each job opening, making it challenging for recruiters to **identify the most suitable candidates quickly**. **Traditional keyword-based search methods often fail to capture the context and meaning of the candidate's skills and experiences**, leading to more **mismatches and unoptimal hiring decisions**.

## **Goals**
The goal of this project is to **build a semantic search system that can analyze and retrieve relevant information from a collection of CVs based on user queries**. The semantic search engine will **leverage natural language processing techniques to understand the context and meaning of the queries**, allowing for **more accurate and relevant search results.**

## **Semantic Search**

[Semantic Search](https://sbert.net/examples/sentence_transformer/applications/semantic-search/README.html) is an information retrieval technique that finds relevant results based on the meaning, context, and intent of a user's search query, rather than relying solely on exact keyword matching.

It uses Natural Language Processing (NLP) and machine learning models to understand the semantic relationships between words, sentences, and documents. Modern semantic search systems commonly use numerical representations called embeddings to identify and retrieve semantically similar information.

## **Semantic Search vs. Keyword Search**

For example, a recruiter enters this query:
"Find candidates with experience in developing business intelligence dashboards and analyzing business performance."

Consider these CV records:
| Candidate | CV Content |
| :--- | :--- |
| **A** | Developed Power BI dashboards and monitored business KPIs. |
| **B** | Experienced in recruitment and talent acquisition. |
| **C** | Created data visualizations and generated business insights. |

- **Keyword Search**: Primarily retrieves documents containing matching words, potentially missing relevant CVs that use different terminology.
- **Semantic Search**: Identifies conceptually related content, potentially retrieving Candidate A and C because their experience is semantically related to the query. This ability to **retrieve relevant information despite differences in wording** is one of the main advantages of semantic search.

### **Differences Between Keyword Search and Semantic Search**

| Keyword Search              | Semantic Search                        |
| --------------------------- | -------------------------------------- |
| Matches **exact words**     | Matches **meaning**                    |
| Sensitive to spelling/order | Understands **context**                |
| No synonyms or paraphrases  | Recognizes synonyms and reworded ideas |
| e.g. “CEO advice”           | Will find: “tips for business leaders” |

## **Embeddings and Models**
For generating sentence representations, we rely on pre-trained transformer networks such as:
`all-MiniLM-L6-v2` from [Sentence-Transformers](https://sbert.net/examples/sentence_transformer/applications/semantic-search/README.html) – a compact and efficient model tailored for semantic search tasks.

This model transform raw text into numerical vector arrays that capture semantic meaning, contextual nuances, and lexical interconnections.

## **Method to Evaluate Semantic Search:**

- **Cosine Similarity** – an intuitive method used to evaluate the distance between the query vector and target text vectors, ordering them by closeness.
- This initiative will leverage sentence-transformers to convert text inputs into numerical embeddings, evaluated collectively through cosine similarity.

# **Data Understanding**

### **Load and read dataset**

Dataset Source: [Kaggle Dataset](https://www.kaggle.com/datasets/haidermaseeh/resume-dataset)

In [60]:
# load data

df = pd.read_csv("resume-dataset.csv")

df.head()

,category,job_title,Text
0,Java Developers/Architects Resumes,Java Developer,Flair for technology and willing to take chall...
1,Java Developers/Architects Resumes,Sr. Java/J2EE Developer,Senior Java Developer with around 7+ years of ...
2,Java Developers/Architects Resumes,Sr. iOS Developer,Overall 7 years of experience in design and de...
3,Java Developers/Architects Resumes,Core Java Developer,7 + years of experience in Application Analysi...
4,Java Developers/Architects Resumes,Android Developer,With approx 5 years of experience in IT sector...


In [61]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9000 entries, 0 to 8999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   category   9000 non-null   str  
 1   job_title  9000 non-null   str  
 2   Text       9000 non-null   str  
dtypes: str(3)
memory usage: 101.3 MB


- There are 9000 non null records in the dataset. The dataset contains 3 columns: Category, Job Title and Text. The text contains the text of the candidate's CV, which will be used for semantic search.

# **Data Cleaning and Preparation**

**Handle Duplicate Values**

In [62]:
# check duplicates

print(f'Duplicated values = {df.duplicated().sum()}')

Duplicated values = 2156


There are duplicated values in the dataset

In [63]:
# drop duplicates
df = df.drop_duplicates()

# check duplicates

print(f'Duplicated values = {df.duplicated().sum()}')

Duplicated values = 0


The duplicated values are dropped from the dataset to ensure that the semantic search results are not biased or skewed by repeated entries. After cleaning, the dataset is ready for further processing and analysis.

In [64]:
# check shape of the dataset after cleaning

print(f'Shape of the dataset after cleaning = {df.shape}')

Shape of the dataset after cleaning = (6844, 3)


**Handle Inconsistent Data**

In [65]:
# replace name of the columns to make it align with other columns

df.rename(columns={'Text': 'text'}, inplace=True)

df.head()

,category,job_title,text
0,Java Developers/Architects Resumes,Java Developer,Flair for technology and willing to take chall...
1,Java Developers/Architects Resumes,Sr. Java/J2EE Developer,Senior Java Developer with around 7+ years of ...
2,Java Developers/Architects Resumes,Sr. iOS Developer,Overall 7 years of experience in design and de...
3,Java Developers/Architects Resumes,Core Java Developer,7 + years of experience in Application Analysi...
4,Java Developers/Architects Resumes,Android Developer,With approx 5 years of experience in IT sector...


In [66]:
# add ID to the resume in dataset

df['id'] = df.index + 1

# rearrange the columns to have ID as the first column
df = df[['id', 'category', 'job_title', 'text']]

df.head()

,id,category,job_title,text
0,1,Java Developers/Architects Resumes,Java Developer,Flair for technology and willing to take chall...
1,2,Java Developers/Architects Resumes,Sr. Java/J2EE Developer,Senior Java Developer with around 7+ years of ...
2,3,Java Developers/Architects Resumes,Sr. iOS Developer,Overall 7 years of experience in design and de...
3,4,Java Developers/Architects Resumes,Core Java Developer,7 + years of experience in Application Analysi...
4,5,Java Developers/Architects Resumes,Android Developer,With approx 5 years of experience in IT sector...


**Save the cleaned dataset** 
- This ensures that the cleaned data is preserved and can be easily accessed for subsequent analysis or model training.

In [67]:
df.to_csv('resume-dataset-cleaned.csv', index=False)

In [68]:
# Detail info of every variables

list_item = []

for col in df.columns:
    list_item.append([
        col, round((df[col].isna().sum() / len(df[col])) * 100, 2), df[col].drop_duplicates().values
    ])

df_desc = pd.DataFrame(
    columns=['variables', 'null (%)', 'unique values'],
    data=list_item
)

df_desc

,variables,null (%),unique values
0,id,0.0,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
1,category,0.0,"[Java Developers/Architects Resumes, Web Devel..."
2,job_title,0.0,"[Java Developer, Sr. Java/J2EE Developer, Sr. ..."
3,text,0.0,[Flair for technology and willing to take chal...


# **Semantic Search Modelling**

### **Load Pretrained Models**

**Sentence Transformers** are fine-tuned to capture the semantic meaning of sentences. <br>

In this project, we will use a popular, small but effective model: **all-MiniLM-L6-v2**

The advantages of this model include:
- Fast and lightweight (great for real-time apps)
- Multilingual
- Trained for semantic similarity tasks



In [69]:
# Load Pre-trained Model
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1132.61it/s]


### **Encode Text into Embeddings**

In [70]:
# Get the resume texts as a list
resume_texts = df['text'].tolist()

# Encode text into dense vector representations
resume_embeddings = model.encode(resume_texts, convert_to_tensor=True)

# **convert to tensor=True** is used to convert the embeddings into PyTorch tensors
# convert to tensor can optimize for GPU acceleration and efficient computation. this is particularly useful when working with large datasets or performing similarity calculations.

In [71]:
# resume embeddings shape
print(f"Resume embeddings shape: {resume_embeddings.shape}")

Resume embeddings shape: torch.Size([6844, 384])


### **Encode the User Query into Embeddings**

In [72]:
query = "I need someone to design and govern data architectures, implement cloud data warehouses, and establish robust data modeling frameworks."
query_embedding = model.encode(query, convert_to_tensor=True)

In [73]:
# query embeddings shape
print(f"Query embeddings shape: {query_embedding.shape}")

Query embeddings shape: torch.Size([384])


## **Cosine Similarity**

### 📐 Understanding **Cosine Similarity**

Cosine similarity is a mathematical metric used to evaluate **how closely aligned two vectors are in orientation**, regardless of their magnitude (length).

When working with NLP models like `all-MiniLM-L6-v2`, sentences, quotes, or search queries are mapped into high-dimensional vector representations (e.g., 384 dimensions). Cosine similarity measures the **angular similarity** between these directional vectors in multi-dimensional space.

---

### 🧮 Mathematical Formulation

Given two multidimensional vectors $\mathbf{A}$ and $\mathbf{B}$, the similarity score is calculated as:

$$\text{similarity}(\mathbf{A}, \mathbf{B}) = \frac{\mathbf{A} \cdot \mathbf{B}}{\Vert{}\mathbf{A}\Vert{} \Vert{}\mathbf{B}\Vert{}}$$

Where:

* $\mathbf{A} \cdot \mathbf{B}$ denotes the **dot product** of vectors $\mathbf{A}$ and $\mathbf{B}$.
* $\Vert{}\mathbf{A}\Vert{}$ and $\Vert{}\mathbf{B}\Vert{}$ represent the **Euclidean norms (lengths)** of each vector.

---

### 📊 Score Interpretation

| Output Value | Semantic Relationship |
| --- | --- |
| **`1.0`** | **Identical Orientation** — The vectors point in the exact same direction (highest similarity). |
| **`0.0`** | **Orthogonal / Uncorrelated** — No shared semantic relationship between the vectors. |
| **`-1.0`** | **Diametrically Opposite** — The vectors point in inverse directions (rarely occurs in dense text embeddings). |

#### Application in Semantic Search:

* The **search query** is encoded into a prompt vector.
* Each candidate **CV document text** is converted into a target vector.
* Cosine similarity determines **how closely the context and intent of the document match the query**.

---

### **Why Cosine Similarity is Preferred for Semantic Search?**
#### ⚖️ Cosine Similarity vs. Euclidean Distance

| Metric | Cosine Similarity | Euclidean Distance ($L_2$) |
| --- | --- | --- |
| **Primary Focus** | **Angular Direction** | Distance & Magnitude |
| **Magnitude Sensitivity** | ❌ **Invariant** (normalizes vector magnitude) | ✅ **Sensitive** (longer texts can inflate distance) |
| **Primary Use Case** | **Dense Text Embeddings** & Semantic Search | Spatial Data, K-Means Clustering, Physical Coordinates |

### **Calculate the Similarity Score with Cosine Similarity Metric**

In [79]:
# Compute cosine similarity scores between query and all resumes
cosine_scores = util.cos_sim(query_embedding, resume_embeddings)[0]

# Get top 10 most similar resumes with query
top_results = cosine_scores.topk(k=5)

In [75]:
print(f"\nTop resumes for: '{query}'\n")
for score, idx in zip(top_results[0], top_results[1]):
    idx = idx.item()
    resume = df.iloc[idx]['job_title']
    text = df.iloc[idx]['text']
    print(f"{score:.4f} — {resume} - ({text})")


Top resumes for: 'I need someone to design and govern data architectures, implement cloud data warehouses, and establish robust data modeling frameworks.'

0.5984 — Sr. Informatica Developer - (Have nearly 10 years of Developer experience in Data warehouse & Business Intelligence platform using Informatica Power Center, Informatica Cloud, Salesforce.com, Informatica Data Quality, Oracle, Teradata, SQL Server, UNIX/Linux, Shell scripting, Business Objects, Cognos Reports, Pentaho Data Integration (Kettle), Python. Worked in Healthcare, Health insurance, Banking/Financial, Manufacturing domains. Expert in the Data warehouse concepts and methodologies such as Ralph Kimball and Bill Inmon models Have extensively worked on OLTP, OLAP, Dimensional modeling - Star schema, Snowflake schema etc, Operational Data store, Conceptual Data Base Designing,B2B exchange, Web services, SOA architecture Certified in Informatica mapping designer, Administration & Architecture, Informatica Data Quality an

# **Model and Data Limitations & Future Improvements**

### 1. Model Limitations
- **Limited Context Understanding:** The `all-MiniLM-L6-v2` model may not fully understand complex sentences, specific technical terms, or the relationship between different skills and work experiences.
- **Similarity Score:** A high similarity score does not necessarily mean that a candidate is qualified for a job. It only indicates how closely the resume content matches the search query in terms of meaning.
- **Long Resume Content:** Converting an entire resume into one embedding may cause important information to become less noticeable when the resume contains many different experiences and skills.

### 2. Dataset Limitations
- **Limited Dataset Size:** A small number of resumes (9000 entries) and only 4 columns (id, category, job_title, text) in this dataset may cause the system to miss relevant candidates who are not included in the dataset and may not provide enough information for accurate semantic search.
- **Incomplete Information:** Some resumes may have missing information about skills, work experience, education, or certifications.
- **Uneven Data Distribution:** Some job categories may have more resumes than others, making search results less balanced.

## Future Improvements

### 1. Dataset Improvements
- **Expand the Dataset:** Add more resumes from different industries, job roles, and experience levels to improve search coverage.
- **Add More Information:** Include additional detail columns such as technical skills, years of experience, education, certifications, and job categories to make search results more relevant.
- **Balance Job Categories:** Include more resumes from job categories that currently have fewer records.

### 2. Model Improvements
- **Test Other Models:** Compare `all-MiniLM-L6-v2` with other embedding models, such as `all-mpnet-base-v2` or `BAAI/bge-base-en-v1.5`, to find a model that provides better search results.
- **Evaluate Search Accuracy:** Test search results using different queries and manually check whether the retrieved resumes are actually relevant.